# ML Training Pipeline

Binary classification with feature engineering, model training, evaluation, and MLflow logging.

## Setup

In [ ]:
hello
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

RANDOM_STATE = 42
print('Setup complete')

## Generate Dataset

In [ ]:
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    random_state=RANDOM_STATE,
)
feature_names = [f'feature_{i:02d}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f'Dataset: {df.shape[0]} rows, {df.shape[1]-1} features')
print(f'Class balance: {df["target"].value_counts().to_dict()}')

## Feature Engineering

In [ ]:
# Add interaction and polynomial features
df['feat_ratio'] = df['feature_00'] / (df['feature_01'].abs() + 1e-6)
df['feat_sum']   = df[['feature_02', 'feature_03', 'feature_04']].sum(axis=1)
df['feat_sq']    = df['feature_05'] ** 2

feature_cols = [c for c in df.columns if c != 'target']
print(f'Total features after engineering: {len(feature_cols)}')

## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols], df['target'],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['target'],
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

## Train Models

In [ ]:
models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':          RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_sc)[:, 1])
    results[name] = {'model': model, 'auc': auc}
    print(f'{name:30s}  AUC: {auc:.4f}')

## Evaluate Best Model

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots()
im = ax.imshow(cm, cmap='Blues')
ax.set(xticks=[0,1], yticks=[0,1],
       xticklabels=['Pred 0','Pred 1'],
       yticklabels=['True 0','True 1'],
       title=f'Confusion Matrix — {best_name}')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout()
plt.show()

In [ ]:
best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}  (AUC {results[best_name]["auc"]:.4f})\n')

y_pred = best_model.predict(X_test_sc)
print(classification_report(y_test, y_pred))

## Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)[:15]
    imp.plot(kind='barh', title='Top 15 Feature Importances', color='teal')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('Feature importances not available for this model type.')

## Summary

| Model | AUC |
|---|---|
| Logistic Regression | — |
| Random Forest | — |
| Gradient Boosting | — |

> Run all cells to populate the table above with live results.